### 08. Climate Dataset Harmonization: NUTS-2 to NUTS-3 (Kreise)

Spatial Harmonization of Climate Indicators for District-Level Climate & ESG Risk Modelling

#### Data Inputs

This notebook uses the following input datasets:

- Harmonized NUTS-2 climate dataset
- Germany NUTS-2 administrative boundaries
- Germany NUTS-3 (Kreise) administrative boundaries

**Objective**

This notebook harmonizes climate indicators from the NUTS-2 regional level to the finer NUTS-3 (Kreise) district level in Germany using area-weighted spatial interpolation.

**Why do this**

The climate risk modeling framework requires climate indicators at the district (Kreise / NUTS-3) level.

Because the climate indicators are available only at the NUTS-2 regional level, they must be spatially disaggregated to NUTS-3 before they can be integrated with district-level socio-economic indicators for climate and ESG risk modelling.

**Approach**

1. Load and validate the NUTS-2 climate dataset.
2. Load the NUTS-2 and NUTS-3 administrative boundaries.
3. Ensure both datasets share a common coordinate reference system (CRS).
4. Compute spatial intersections and area-weighting factors.
5. Disaggregate climate indicators from NUTS-2 to NUTS-3 using area-weighted interpolation.
6. Validate spatial coverage and aggregation consistency.
7. Export the harmonized NUTS-3 climate dataset.

**Outputs**

- Spatial Crosswalk (NUTS-2 → NUTS-3)
- Area-weight interpolation factors
- Harmonized NUTS-3 climate datasets
- Spatial validation and quality assessment diagnostics

# Part I — Build and Validate the Spatial Crosswalk (Area-Weight Table)

### Import Libraries

In [8]:
# ============================================================
# 0. Import libraries and settings
# ============================================================

import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from pathlib import Path
from shapely.geometry import Polygon

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

BASE_DIR = Path.cwd().parent          # Project root
RAW_CLIMATE_DIR = BASE_DIR / "data" / "raw" / "climate"

print("Project directory :", BASE_DIR)
print("Climate data path :", RAW_CLIMATE_DIR)
print("Path exists       :", RAW_CLIMATE_DIR.exists())

Project directory : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany
Climate data path : d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate
Path exists       : True


##### Create Germany-only NUTS Boundary Files

In [11]:
# ============================================================
# 1. Create Germany-only NUTS boundary files
# ============================================================

# Input Europe-wide NUTS boundary files
nuts2_europe_path = RAW_CLIMATE_DIR / "germany" / "NUTS_RG_01M_2021_4326_LEVL_2.shp"
nuts3_europe_path = RAW_CLIMATE_DIR / "germany" / "NUTS_RG_01M_2021_4326_LEVL_3.shp"

# Output Germany-only NUTS boundary files
nuts2_de_path = RAW_CLIMATE_DIR / "germany" / "nuts2_germany_2021.shp"
nuts3_de_path = RAW_CLIMATE_DIR / "germany" / "nuts3_germany_2021.shp"

# Check that input files exist
print("NUTS-2 Europe file exists:", nuts2_europe_path.exists())
print("NUTS-3 Europe file exists:", nuts3_europe_path.exists())

# Load Europe-wide NUTS files
nuts2 = gpd.read_file(nuts2_europe_path)
nuts3 = gpd.read_file(nuts3_europe_path)

# Filter Germany only
nuts2_de = nuts2[nuts2["CNTR_CODE"] == "DE"].copy()
nuts3_de = nuts3[nuts3["CNTR_CODE"] == "DE"].copy()

# Print diagnostics
print("\nEurope NUTS-2 shape:", nuts2.shape)
print("Germany NUTS-2 shape:", nuts2_de.shape)

print("\nEurope NUTS-3 shape:", nuts3.shape)
print("Germany NUTS-3 shape:", nuts3_de.shape)

print("\nNUTS-2 CRS:", nuts2_de.crs)
print("NUTS-3 CRS:", nuts3_de.crs)

# Save Germany-only boundary files
nuts2_de.to_file(nuts2_de_path)
nuts3_de.to_file(nuts3_de_path)

print("\nSaved Germany-only NUTS-2 to:", nuts2_de_path)
print("Saved Germany-only NUTS-3 to:", nuts3_de_path)

NUTS-2 Europe file exists: True
NUTS-3 Europe file exists: True

Europe NUTS-2 shape: (334, 18)
Germany NUTS-2 shape: (38, 18)

Europe NUTS-3 shape: (1514, 18)
Germany NUTS-3 shape: (401, 18)

NUTS-2 CRS: EPSG:4326
NUTS-3 CRS: EPSG:4326

Saved Germany-only NUTS-2 to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate\germany\nuts2_germany_2021.shp
Saved Germany-only NUTS-3 to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\raw\climate\germany\nuts3_germany_2021.shp


##### Load Germany NUTS Administrative Boundaries

In [13]:
# ============================================================
# 2. Load Germany NUTS administrative boundaries
# ============================================================

nuts2_path = RAW_CLIMATE_DIR / "germany" / "nuts2_germany_2021.shp"
nuts3_path = RAW_CLIMATE_DIR / "germany" / "nuts3_germany_2021.shp"

nuts2 = gpd.read_file(nuts2_path)
nuts3 = gpd.read_file(nuts3_path)

print(f"NUTS-2 regions: {len(nuts2)}")
print(f"NUTS-3 districts: {len(nuts3)}")

display(nuts2.head())
display(nuts3.head())

NUTS-2 regions: 38
NUTS-3 districts: 401


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DE12,2,DE,Karlsruhe,Karlsruhe,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((9.45238 49.64864, 9.47674 49.6..."
1,DE14,2,DE,Tübingen,Tübingen,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((9.96528 48.59584, 9.9727 48.59462, 9..."
2,DE13,2,DE,Freiburg,Freiburg,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((8.00179 48.68692, 8.01441 48.6..."
3,DE21,2,DE,Oberbayern,Oberbayern,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((11.42642 49.06797, 11.44318 49..."
4,DEF0,2,DE,Schleswig-Holstein,Schleswig-Holstein,0.0,0,0,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((8.44242 55.05062, 8.46241 55.0..."


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DE933,3,DE,Harburg,Harburg,4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((9.78358 53.5, 9.78321 53.49212..."
1,DE934,3,DE,Lüchow-Dannenberg,Lüchow-Dannenberg,4,3,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((10.97163 53.1981, 11.01166 53.18148,..."
2,DE935,3,DE,"Lüneburg, Landkreis","Lüneburg, Landkreis",4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"MULTIPOLYGON (((10.55863 53.36796, 10.57365 53..."
3,DEA5A,3,DE,Siegen-Wittgenstein,Siegen-Wittgenstein,2,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((8.49703 51.13709, 8.50502 51.13271, ..."
4,DEA5B,3,DE,Soest,Soest,4,2,3,Germany,Allemagne,DEU,UN Member State,Berlin,T,F,F,Deutschland,"POLYGON ((8.33538 51.72791, 8.34204 51.72319, ..."


##### Inspect Boundary Attributes

In [15]:
# ============================================================
# 3. Inspect boundary attributes
# ============================================================

print(f"NUTS-2 regions: {len(nuts2)}")
print(f"NUTS-3 districts: {len(nuts3)}")

print("\nCoordinate Reference System (CRS)")
print("NUTS-2:", nuts2.crs)
print("NUTS-3:", nuts3.crs)

print("\nKey attributes")
print("NUTS-2:", ["NUTS_ID", "LEVL_CODE", "CNTR_CODE", "NUTS_NAME", "geometry"])
print("NUTS-3:", ["NUTS_ID", "LEVL_CODE", "CNTR_CODE", "NUTS_NAME", "geometry"])

print("\nCountry codes")
print("NUTS-2:", nuts2["CNTR_CODE"].unique())
print("NUTS-3:", nuts3["CNTR_CODE"].unique())

NUTS-2 regions: 38
NUTS-3 districts: 401

Coordinate Reference System (CRS)
NUTS-2: EPSG:4326
NUTS-3: EPSG:4326

Key attributes
NUTS-2: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NUTS_NAME', 'geometry']
NUTS-3: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NUTS_NAME', 'geometry']

Country codes
NUTS-2: ['DE']
NUTS-3: ['DE']


## Build the Spatial Crosswalk (NUTS-2 → NUTS-3 Area Weights)

This step constructs the **Spatial Crosswalk (NUTS-2 → NUTS-3)** by calculating area-weight interpolation factors between Germany's NUTS-2 regions and NUTS-3 districts. The resulting area-weight table provides the spatial translation layer required to harmonize climate indicators from the NUTS-2 regional level to the NUTS-3 district level in a transparent and reproducible manner.

The resulting spatial crosswalk serves as the foundation for harmonizing all climate indicators that are currently available only at the NUTS-2 regional level. By calculating the proportion of each NUTS-3 district that overlaps with a NUTS-2 region, we derive area-weighting factors that are subsequently used to disaggregate regional climate indicators to the district level.

### Methodological Justification

Area-weight interpolation was selected because the climate indicators used in this project are available as aggregated regional statistics at the NUTS-2 level, while the target modelling framework operates at the finer NUTS-3 district level. In the absence of higher-resolution climate observations, the spatial crosswalk provides a transparent, reproducible, and computationally efficient approach for spatial harmonization.

### Methodological Assumptions

The harmonization assumes that each climate indicator represents a regional average that is spatially homogeneous within each NUTS-2 region. Consequently, the climate value assigned to a NUTS-3 district is estimated using the proportional area of overlap between the district and its parent NUTS-2 region.

The resulting area-weight table is generated once and subsequently reused for all climate indicators, ensuring methodological consistency across heatwave, flood, and drought preprocessing workflows.

In [55]:
# ============================================================
# Construct the Spatial Crosswalk (NUTS-2 -> NUTS-3)
# ============================================================

# Use a projected CRS for accurate area calculations.
# EPSG:3035 is commonly used for Europe-wide equal-area analysis.
nuts2_proj = nuts2.to_crs(epsg=3035)
nuts3_proj = nuts3.to_crs(epsg=3035)

print("Projected CRS")
print("NUTS-2:", nuts2_proj.crs)
print("NUTS-3:", nuts3_proj.crs)

# Keep only required columns before overlay
nuts2_small = nuts2_proj[["NUTS_ID", "NUTS_NAME", "geometry"]].rename(
    columns={
        "NUTS_ID": "nuts2_id",
        "NUTS_NAME": "nuts2_name"
    }
)

nuts3_small = nuts3_proj[["NUTS_ID", "NUTS_NAME", "geometry"]].rename(
    columns={
        "NUTS_ID": "nuts3_id",
        "NUTS_NAME": "nuts3_name"
    }
)

# Calculate original NUTS-3 district area
nuts3_area = nuts3_small.copy()
nuts3_area["nuts3_area_m2"] = nuts3_area.geometry.area
nuts3_area = nuts3_area[["nuts3_id", "nuts3_area_m2"]]

# Compute spatial intersections required to construct the Spatial Crosswalk
intersections = gpd.overlay(
    nuts3_small,
    nuts2_small,
    how="intersection",
    keep_geom_type=False
)

# Calculate overlap area
intersections["overlap_area_m2"] = intersections.geometry.area

# Attach original NUTS-3 area and calculate area weight
area_weights = intersections.merge(nuts3_area, on="nuts3_id", how="left")
area_weights["area_weight"] = (
    area_weights["overlap_area_m2"] / area_weights["nuts3_area_m2"]
)

# Retain the final Spatial Crosswalk columns
area_weights = area_weights[
    [
        "nuts2_id",
        "nuts2_name",
        "nuts3_id",
        "nuts3_name",
        "overlap_area_m2",
        "nuts3_area_m2",
        "area_weight"
    ]
].copy()

# Sort the Spatial Crosswalk for reproducibility
area_weights = area_weights.sort_values(["nuts2_id", "nuts3_id"]).reset_index(drop=True)

print("Spatial Crosswalk (NUTS-2 → NUTS-3) created successfully.")
print("Spatial Crosswalk dimensions (rows, columns):", area_weights.shape)
print("Preview of the validated Spatial Crosswalk:")

display(area_weights.head())

Projected CRS
NUTS-2: EPSG:3035
NUTS-3: EPSG:3035
Spatial Crosswalk (NUTS-2 → NUTS-3) created successfully.
Spatial Crosswalk dimensions (rows, columns): (865, 7)
Preview of the validated Spatial Crosswalk:


,nuts2_id,nuts2_name,nuts3_id,nuts3_name,overlap_area_m2,nuts3_area_m2,area_weight
0,DE11,Stuttgart,DE111,"Stuttgart, Stadtkreis",2.101117e+08,2.101117e+08,1.0
1,DE11,Stuttgart,DE112,Böblingen,6.167061e+08,6.167061e+08,1.0
2,DE11,Stuttgart,DE113,Esslingen,6.396529e+08,6.396529e+08,1.0
3,DE11,Stuttgart,DE114,Göppingen,6.421750e+08,6.421750e+08,1.0
4,DE11,Stuttgart,DE115,Ludwigsburg,6.859905e+08,6.859905e+08,1.0


##### Initial Validation of the Spatial Crosswalk

In [56]:
# ============================================================
# 5a. Validate the initial Spatial Crosswalk (NUTS-2 -> NUTS-3)
# ============================================================

district_counts = (
    area_weights
    .groupby("nuts3_id")
    .size()
    .value_counts()
    .sort_index()
)

print(district_counts)

1    116
2    148
3    100
4     32
5      5
Name: count, dtype: int64


##### Inspect districts with multiple NUTS-2 intersections

In [57]:
# ============================================================
# 5b. Inspect districts with multiple NUTS-2 intersections
# ============================================================

multiple = (
    area_weights
    .groupby("nuts3_id")
    .filter(lambda x: len(x) > 1)
)

display(
    multiple.sort_values(["nuts3_id", "area_weight"])
)

,nuts2_id,nuts2_name,nuts3_id,nuts3_name,overlap_area_m2,nuts3_area_m2,area_weight
29,DE12,Karlsruhe,DE112,Böblingen,0.000000e+00,6.167061e+08,0.0
73,DE14,Tübingen,DE112,Böblingen,0.000000e+00,6.167061e+08,0.0
1,DE11,Stuttgart,DE112,Böblingen,6.167061e+08,6.167061e+08,1.0
74,DE14,Tübingen,DE113,Esslingen,0.000000e+00,6.396529e+08,0.0
2,DE11,Stuttgart,DE113,Esslingen,6.396529e+08,6.396529e+08,1.0
...,...,...,...,...,...,...,...
862,DEG0,Thüringen,DEG0M,Altenburger Land,5.698137e+08,5.698137e+08,1.0
400,DE73,Kassel,DEG0N,"Eisenach, Kreisfreie Stadt",0.000000e+00,1.046425e+08,0.0
863,DEG0,Thüringen,DEG0N,"Eisenach, Kreisfreie Stadt",1.046425e+08,1.046425e+08,1.0
401,DE73,Kassel,DEG0P,Wartburgkreis,0.000000e+00,1.305336e+09,0.0


**Interpretation**

Inspection of the multiple matches shows that the additional NUTS-2 assignments all have an area weight of 0.0, indicating that they arise solely from shared administrative boundaries rather than genuine spatial overlap. These records can therefore be safely removed without affecting the integrity of the resulting Spatial Crosswalk (NUTS-2 → NUTS-3).

##### Remove Zero-Area Boundary Artefacts

The initial spatial overlay identifies all geometric intersections between NUTS-2 and NUTS-3 polygons. Some of these correspond only to shared administrative boundaries and therefore have zero overlap area. These boundary artefacts are removed before constructing the final Spatial Crosswalk (NUTS-2 → NUTS-3) to ensure that each NUTS-3 district is associated only with its true parent NUTS-2 region.

In [58]:
area_weights = area_weights.query("area_weight > 0").copy()

##### Final Validation of the Spatial Crosswalk

In [59]:
# ============================================================
# 5c. Validate the Spatial Crosswalk
# ============================================================

district_counts = (
    area_weights
    .groupby("nuts3_id")
    .size()
    .value_counts()
    .sort_index()
)

print(district_counts)

1    401
Name: count, dtype: int64


**Validation Outcome**

After removing zero-area boundary artefacts, each of Germany's 401 NUTS-3 districts is associated with exactly one NUTS-2 region. This confirms that the validated Spatial Crosswalk (NUTS-2 → NUTS-3) provides a reliable spatial translation layer for subsequent climate data harmonization.

#### Save the Validated Spatial Crosswalk

The validated **Spatial Crosswalk (NUTS-2 → NUTS-3)** is saved as a reusable project asset. It will be used by all subsequent climate harmonization workflows to transfer climate indicators from the NUTS-2 regional level to Germany's NUTS-3 districts using consistent area-weight interpolation.

In [60]:
# ==========================================================
# 7. Save the validated Spatial Crosswalk (NUTS-2 -> NUTS-3)
# ==========================================================

OUTPUT_DIR = BASE_DIR / "data" / "processed" / "climate" / "spatial_lookup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

area_weights.to_csv(
    OUTPUT_DIR / "nuts2_to_nuts3_area_weights.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Spatial Crosswalk saved to: {OUTPUT_DIR}")
print(f"Validated Spatial Crosswalk districts: {len(area_weights):,}")

Spatial Crosswalk saved to: d:\Purity\GitHub-Repository\climate-esg-risk-intelligence-germany\data\processed\climate\spatial_lookup
Validated Spatial Crosswalk districts: 401


## Summary

In this notebook, a reusable Spatial Crosswalk (NUTS-2 → NUTS-3) was constructed and validated. The Spatial Crosswalk provides the area-weight interpolation factors required to harmonize climate indicators available at the NUTS-2 regional level to Germany's NUTS-3 districts.

The validated Spatial Crosswalk will be integrated into the climate harmonization workflows, enabling all climate indicators available at the NUTS-2 regional level to be consistently transferred to the NUTS-3 district level. The resulting harmonized climate datasets will provide the spatial foundation for integrating climate hazards with socioeconomic indicators and constructing the integrated Climate-ESG Risk Intelligence modelling dataset.

## Next Step

The validated Spatial Crosswalk (NUTS-2 → NUTS-3) will now be integrated into the heatwave, flood (Rx5day), and drought preprocessing workflows. Each climate indicator available at the NUTS-2 level will be transferred to the NUTS-3 district level using the validated area-weight interpolation approach, providing the spatial foundation for the integrated Climate–ESG Risk Intelligence modelling dataset.

# Data Integration Compatibility Check

### Integration Compatibility Check
### GISCO NUTS-3 vs. INKAR AGS Districts

Before harmonizing the climate datasets, verify that the GISCO NUTS-3 administrative units can be reliably integrated with the INKAR socio-economic datasets. This diagnostic identifies whether both data sources share a common district identifier or whether an intermediate Administrative Crosswalk (AGS → NUTS-3) is required.

##### Inspect the GISCO NUTS-3 Reference Table

In [61]:
# ================================================================
# Integration Check 1
# GISCO NUTS-3 Reference Table
# ================================================================

print("Rows:", len(area_weights))

print("\nColumns:")
print(area_weights.columns.tolist())

print("\nUnique NUTS-3 IDs:", area_weights["nuts3_id"].nunique())

display(
    area_weights[
        ["nuts3_id", "nuts3_name"]
    ].drop_duplicates().head(10)
)

Rows: 401

Columns:
['nuts2_id', 'nuts2_name', 'nuts3_id', 'nuts3_name', 'overlap_area_m2', 'nuts3_area_m2', 'area_weight']

Unique NUTS-3 IDs: 401


,nuts3_id,nuts3_name
0,DE111,"Stuttgart, Stadtkreis"
1,DE112,Böblingen
2,DE113,Esslingen
3,DE114,Göppingen
4,DE115,Ludwigsburg
5,DE116,Rems-Murr-Kreis
6,DE117,"Heilbronn, Stadtkreis"
7,DE118,"Heilbronn, Landkreis"
8,DE119,Hohenlohekreis
9,DE11A,Schwäbisch Hall


In [41]:
# ============================================================
# Raw socioeconomic datasets
# ============================================================

RAW_SOCIOECONOMIC_DIR = (
    BASE_DIR /
    "data" /
    "raw" /
    "socioeconomic"
)

##### Load one INKAR dataset

In [62]:
RAW_SOCIOECONOMIC_DIR = BASE_DIR / "data" / "raw" / "socioeconomic"

unemployment = pd.read_csv(
    RAW_SOCIOECONOMIC_DIR / "unemployment_inkar_1998_2023.csv"
)

print(unemployment.columns.tolist())
display(unemployment.head())
print(unemployment.shape)

['district_code', 'district_name', 'aggregation_level', 'year', 'unemployment_rate']


,district_code,district_name,aggregation_level,year,unemployment_rate
0,1001,"Flensburg, Stadt",Kreise,1998,14.60
1,1002,"Kiel, Landeshauptstadt",Kreise,1998,13.90
2,1003,"Lübeck, Hansestadt",Kreise,1998,13.41
3,1004,"Neumünster, Stadt",Kreise,1998,13.45
4,1051,Dithmarschen,Kreise,1998,10.70


(10400, 5)


##### Compare Identifiers / district names

In [63]:
# ============================================================
# Compare GISCO and INKAR identifiers
# ============================================================

print("INKAR columns:")
print(unemployment.columns.tolist())

print("\nUnique district codes:")
print(unemployment["district_code"].nunique())

display(
    unemployment[
        ["district_code", "district_name"]
    ].drop_duplicates().head(10)
)

INKAR columns:
['district_code', 'district_name', 'aggregation_level', 'year', 'unemployment_rate']

Unique district codes:
400


,district_code,district_name
0,1001,"Flensburg, Stadt"
1,1002,"Kiel, Landeshauptstadt"
2,1003,"Lübeck, Hansestadt"
3,1004,"Neumünster, Stadt"
4,1051,Dithmarschen
5,1053,Herzogtum Lauenburg
6,1054,Nordfriesland
7,1055,Ostholstein
8,1056,Pinneberg
9,1057,Plön


In [64]:
print("Minimum district code:", unemployment["district_code"].min())
print("Maximum district code:", unemployment["district_code"].max())

Minimum district code: 1001
Maximum district code: 16077


In [65]:
display(
    unemployment[
        ["district_code", "district_name"]
    ].drop_duplicates().sample(20, random_state=42)
)

,district_code,district_name
209,8327,Tuttlingen
280,9561,"Ansbach, Stadt"
33,3351,Celle
210,8335,Konstanz
93,5558,Coesfeld
84,5366,Euskirchen
329,12060,Barnim
94,5562,Recklinghausen
266,9377,Tirschenreuth
126,6437,Odenwaldkreis


##### Determine whether district_code is the official AGS/Kreisschlüssel

In [66]:
print("Rows:", len(unemployment))
print("Unique district codes:", unemployment["district_code"].nunique())

display(
    unemployment[
        ["district_code", "district_name"]
    ]
    .drop_duplicates()
    .sort_values("district_code")
    .tail(20)
)

Rows: 10400
Unique district codes: 400


,district_code,district_name
380,16053,"Jena, Stadt"
381,16054,"Suhl, Stadt"
382,16055,"Weimar, Stadt"
383,16061,Eichsfeld
384,16062,Nordhausen
385,16063,Wartburgkreis
386,16064,Unstrut-Hainich-Kreis
387,16065,Kyffhäuserkreis
388,16066,Schmalkalden-Meiningen
389,16067,Gotha


##### Compare one known district

For example:

GISCO

DE111
Stuttgart, Stadtkreis

In [67]:
unemployment[
    unemployment["district_name"]
    .str.contains("Stuttgart", case=False, na=False)
].drop_duplicates()

,district_code,district_name,aggregation_level,year,unemployment_rate
178,8111,"Stuttgart, Stadtkreis",Kreise,1998,8.16
578,8111,"Stuttgart, Stadtkreis",Kreise,1999,7.32
978,8111,"Stuttgart, Stadtkreis",Kreise,2000,6.18
1378,8111,"Stuttgart, Stadtkreis",Kreise,2001,5.64
1778,8111,"Stuttgart, Stadtkreis",Kreise,2002,6.20
2178,8111,"Stuttgart, Stadtkreis",Kreise,2003,7.06
2578,8111,"Stuttgart, Stadtkreis",Kreise,2004,7.20
2978,8111,"Stuttgart, Stadtkreis",Kreise,2005,9.88
3378,8111,"Stuttgart, Stadtkreis",Kreise,2006,8.26
3778,8111,"Stuttgart, Stadtkreis",Kreise,2007,6.56


In [68]:
unemployment[
    unemployment["district_name"]
    .str.contains("Heilbronn", case=False, na=False)
].drop_duplicates()

,district_code,district_name,aggregation_level,year,unemployment_rate
184,8121,"Heilbronn, Stadtkreis",Kreise,1998,9.81
185,8125,Heilbronn,Kreise,1998,6.74
584,8121,"Heilbronn, Stadtkreis",Kreise,1999,9.11
585,8125,Heilbronn,Kreise,1999,6.14
984,8121,"Heilbronn, Stadtkreis",Kreise,2000,7.70
985,8125,Heilbronn,Kreise,2000,4.82
1384,8121,"Heilbronn, Stadtkreis",Kreise,2001,6.97
1385,8125,Heilbronn,Kreise,2001,4.42
1784,8121,"Heilbronn, Stadtkreis",Kreise,2002,7.78
1785,8125,Heilbronn,Kreise,2002,4.86


In [69]:
unemployment[
    unemployment["district_name"]
    .str.contains("Pforzheim", case=False, na=False)
].drop_duplicates()

,district_code,district_name,aggregation_level,year,unemployment_rate
199,8231,"Pforzheim, Stadtkreis",Kreise,1998,9.88
599,8231,"Pforzheim, Stadtkreis",Kreise,1999,9.11
999,8231,"Pforzheim, Stadtkreis",Kreise,2000,7.49
1399,8231,"Pforzheim, Stadtkreis",Kreise,2001,7.15
1799,8231,"Pforzheim, Stadtkreis",Kreise,2002,8.44
2199,8231,"Pforzheim, Stadtkreis",Kreise,2003,9.82
2599,8231,"Pforzheim, Stadtkreis",Kreise,2004,9.89
2999,8231,"Pforzheim, Stadtkreis",Kreise,2005,10.80
3399,8231,"Pforzheim, Stadtkreis",Kreise,2006,10.95
3799,8231,"Pforzheim, Stadtkreis",Kreise,2007,8.12


### Summary

In this integration compatibility assessment, the validated **Spatial Crosswalk (NUTS-2 → NUTS-3)** was compared with the INKAR socioeconomic datasets to verify that both data sources can be integrated into a common Climate–ESG modelling framework.

The inspection confirmed that the INKAR datasets contain the official German district identifier (`district_code`), corresponding to the official Amtlicher Gemeindeschlüssel (AGS/Kreisschlüssel), while the GISCO NUTS-3 reference table provides the corresponding NUTS-3 district identifiers and district names.

Manual validation using multiple districts, including Stuttgart, Heilbronn, and Pforzheim, confirmed that the district names are consistent between both datasets and that the AGS district codes provide a reliable bridge to the GISCO NUTS-3 administrative units.

This confirms that a permanent **Administrative Crosswalk (AGS → NUTS-3)** can be constructed and subsequently reused to integrate all INKAR socioeconomic datasets, including population, GDP per capita, unemployment, and future indicators, with the harmonized climate datasets.

A subsequent integration step will construct and validate the final **Administrative Crosswalk (AGS → NUTS-3)**, including verification of the remaining district count discrepancy between the GISCO and INKAR datasets.

The successful completion of this validation establishes the foundation for building the unified Climate–ESG master dataset.

### Resolve the 401 vs. 400 District Count Difference

This step compares the GISCO NUTS-3 district list with the INKAR district list to identify the source of the one-district difference between the two datasets.

In [70]:
# ============================================================
# Compare GISCO and INKAR district lists
# ============================================================

gisco_districts = (
    area_weights[["nuts3_id", "nuts3_name"]]
    .drop_duplicates()
    .sort_values("nuts3_id")
    .reset_index(drop=True)
)

inkar_districts = (
    unemployment[["district_code", "district_name"]]
    .drop_duplicates()
    .sort_values("district_code")
    .reset_index(drop=True)
)

print("GISCO NUTS-3 districts:", len(gisco_districts))
print("INKAR districts:", len(inkar_districts))

display(gisco_districts.head())
display(inkar_districts.head())

GISCO NUTS-3 districts: 401
INKAR districts: 400


,nuts3_id,nuts3_name
0,DE111,"Stuttgart, Stadtkreis"
1,DE112,Böblingen
2,DE113,Esslingen
3,DE114,Göppingen
4,DE115,Ludwigsburg


,district_code,district_name
0,1001,"Flensburg, Stadt"
1,1002,"Kiel, Landeshauptstadt"
2,1003,"Lübeck, Hansestadt"
3,1004,"Neumünster, Stadt"
4,1051,Dithmarschen


#### Compare district names

In [71]:
# ============================================================
# Compare district names
# ============================================================

gisco_names = set(gisco_districts["nuts3_name"])
inkar_names = set(inkar_districts["district_name"])

only_in_gisco = sorted(gisco_names - inkar_names)
only_in_inkar = sorted(inkar_names - gisco_names)

print("Names only in GISCO:", len(only_in_gisco))
print("Names only in INKAR:", len(only_in_inkar))

print("\nOnly in GISCO:")
print(only_in_gisco)

print("\nOnly in INKAR:")
print(only_in_inkar)

Names only in GISCO: 131
Names only in INKAR: 130

Only in GISCO:
['Amberg, Kreisfreie Stadt', 'Ansbach, Kreisfreie Stadt', 'Ansbach, Landkreis', 'Aschaffenburg, Kreisfreie Stadt', 'Aschaffenburg, Landkreis', 'Augsburg, Kreisfreie Stadt', 'Augsburg, Landkreis', 'Bamberg, Kreisfreie Stadt', 'Bamberg, Landkreis', 'Bayreuth, Kreisfreie Stadt', 'Bayreuth, Landkreis', 'Berlin', 'Bielefeld, Kreisfreie Stadt', 'Bochum, Kreisfreie Stadt', 'Bonn, Kreisfreie Stadt', 'Bottrop, Kreisfreie Stadt', 'Brandenburg an der Havel, Kreisfreie Stadt', 'Braunschweig, Kreisfreie Stadt', 'Bremen, Kreisfreie Stadt', 'Bremerhaven, Kreisfreie Stadt', 'Chemnitz, Kreisfreie Stadt', 'Coburg, Kreisfreie Stadt', 'Coburg, Landkreis', 'Cottbus, Kreisfreie Stadt', 'Darmstadt, Kreisfreie Stadt', 'Delmenhorst, Kreisfreie Stadt', 'Dessau-Roßlau, Kreisfreie Stadt', 'Dillingen a.d. Donau', 'Dortmund, Kreisfreie Stadt', 'Dresden, Kreisfreie Stadt', 'Duisburg, Kreisfreie Stadt', 'Düsseldorf, Kreisfreie Stadt', 'Eisenach, Kreisf

In [73]:
%whos

Variable                      Type            Data/Info
-------------------------------------------------------
BASE_DIR                      WindowsPath     d:\Purity\GitHub-Reposito<...>risk-intelligence-germany
OUTPUT_DIR                    WindowsPath     d:\Purity\GitHub-Reposito<...>ed\climate\spatial_lookup
PROCESSED_CLIMATE_DIR         WindowsPath     d:\Purity\GitHub-Reposito<...>ny\data\processed\climate
PROCESSED_SOCIOECONOMIC_DIR   WindowsPath     d:\Purity\GitHub-Reposito<...>a\processed\socioeconomic
Path                          type            <class 'pathlib.Path'>
Polygon                       type            <class 'shapely.geometry.polygon.Polygon'>
RAW_CLIMATE_DIR               WindowsPath     d:\Purity\GitHub-Reposito<...>-germany\data\raw\climate
RAW_SOCIOECONOMIC_DIR         WindowsPath     d:\Purity\GitHub-Reposito<...>ny\data\raw\socioeconomic
area_weights                  DataFrame       Shape: (401, 7)
district_counts               Series          Shape: (1,

In [74]:
intersections.columns.tolist()

['nuts3_id',
 'nuts3_name',
 'nuts2_id',
 'nuts2_name',
 'geometry',
 'overlap_area_m2']

#### Count GISCO districts by Bundesland

In [75]:
# ==========================================================
# District counts by Bundesland (GISCO NUTS-3)
# ==========================================================

gisco_state_counts = (
    area_weights[["nuts3_id"]]
    .drop_duplicates()
    .assign(state=lambda df: df["nuts3_id"].str[:3])   # e.g. DE1, DE2 ...
    .groupby("state")
    .size()
    .reset_index(name="gisco_districts")
    .sort_values("state")
)

display(gisco_state_counts)

print("Total GISCO districts:", gisco_state_counts["gisco_districts"].sum())

,state,gisco_districts
0,DE1,44
1,DE2,96
2,DE3,1
3,DE4,18
4,DE5,2
5,DE6,1
6,DE7,26
7,DE8,8
8,DE9,45
9,DEA,53


Total GISCO districts: 401


#### Count INKAR districts by Bundesland

In [76]:
# ==========================================================
# District counts by Bundesland (INKAR AGS)
# ==========================================================

inkar_state_counts = (
    unemployment[["district_code"]]
    .drop_duplicates()
    .assign(
        state=lambda df: (
            df["district_code"]
            .astype(str)
            .str.zfill(5)
            .str[:2]
        )
    )
    .groupby("state")
    .size()
    .reset_index(name="inkar_districts")
    .sort_values("state")
)

display(inkar_state_counts)

print("Total INKAR districts:", inkar_state_counts["inkar_districts"].sum())

,state,inkar_districts
0,01,15
1,02,1
2,03,45
3,04,2
4,05,53
5,06,26
6,07,36
7,08,44
8,09,96
9,10,6


Total INKAR districts: 400


#### Compare the two 

In [77]:
# ==========================================================
# Compare GISCO vs INKAR district counts by Bundesland
# ==========================================================

comparison = (
    gisco_state_counts.merge(
        inkar_state_counts,
        left_on="state",
        right_on="state",
        how="outer"
    )
    .fillna(0)
)

comparison["difference"] = (
    comparison["gisco_districts"] - comparison["inkar_districts"]
)

display(comparison)

,state,gisco_districts,inkar_districts,difference
0,01,0.0,15.0,-15.0
1,02,0.0,1.0,-1.0
2,03,0.0,45.0,-45.0
3,04,0.0,2.0,-2.0
4,05,0.0,53.0,-53.0
5,06,0.0,26.0,-26.0
6,07,0.0,36.0,-36.0
7,08,0.0,44.0,-44.0
8,09,0.0,96.0,-96.0
9,10,0.0,6.0,-6.0


### Standardize District Names

The apparent differences between the GISCO and INKAR district lists are expected because the two datasets use different naming conventions (e.g., "Landkreis", "kreisfreie Stadt", "Stadt").

To identify the true district discrepancy (401 vs. 400), both district name lists are standardized before comparison by removing administrative suffixes and formatting differences.

#### Create a standardization function

In [85]:
# ==========================================================
# Standardize district names
# ==========================================================

import re

def clean_district_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    # Normalize common German abbreviations before punctuation cleanup
    replacements = {
        "a.d.": "an der",
        "a. d.": "an der",
        "i.d.": "in der",
        "i. d.": "in der",
        "i.breisgau": "im breisgau",
        "i. breisgau": "im breisgau",
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    # Remove honorary / descriptive titles
    patterns = [
        r"\blandeshauptstadt\b",
        r"\bwissenschaftsstadt\b",
        r"\bklingenstadt\b",
        r"\bder fernuniversität\b",
        r"\bdocumenta[-\s]?stadt\b",
        r"\bdocumenta\b",
        r"\bhansestadt\b",
        r"\bkreisfreie stadt\b",
        r"\bstadtkreis\b",
        r"\blandkreis\b",
        r"\bstadt\b",
        r"\(de\)",
    ]

    for p in patterns:
        name = re.sub(p, "", name)

    # Remove punctuation after abbreviation normalization
    name = re.sub(r"[.,()\-]", " ", name)

    # Expand compressed German location expressions
    name = re.sub(r"ander", "an der ", name)
    name = re.sub(r"inder", "in der ", name)
    
    # Normalize Hamburg official name
    name = name.replace("freie und", "")

    # Collapse duplicate spaces
    name = re.sub(r"\s+", " ", name).strip()

    return name

#### Aply the Standardization Function

In [86]:
gisco_clean = (
    gisco_districts.assign(
        clean_name=gisco_districts["nuts3_name"].apply(clean_district_name)
    )
)

inkar_clean = (
    inkar_districts.assign(
        clean_name=inkar_districts["district_name"].apply(clean_district_name)
    )
)

gisco_names = set(gisco_clean["clean_name"])
inkar_names = set(inkar_clean["clean_name"])

only_gisco = sorted(gisco_names - inkar_names)
only_inkar = sorted(inkar_names - gisco_names)

print("Only in GISCO:", len(only_gisco))
print("Only in INKAR:", len(only_inkar))

print("\nGISCO only:")
print(only_gisco)

print("\nINKAR only:")
print(only_inkar)

Only in GISCO: 7
Only in INKAR: 6

GISCO only:
['dillingen an der donau', 'eisenach', 'neumarkt in der opf', 'neustadt an der aisch bad windsheim', 'neustadt an der waldnaab', 'pfaffenhofen an der ilm', 'weiden in der opf']

INKAR only:
['dillingen an derdonau', 'neumarkt in deropf', 'neustadt an deraisch bad windsheim', 'neustadt an derwaldnaab', 'pfaffenhofen an derilm', 'weiden in deropf']


#### Compare the two again

In [80]:
# ==========================================================
# Compare standardized names
# ==========================================================

gisco_names = set(gisco_clean["clean_name"])
inkar_names = set(inkar_clean["clean_name"])

only_gisco = sorted(gisco_names - inkar_names)
only_inkar = sorted(inkar_names - gisco_names)

print("Only in GISCO:", len(only_gisco))
print("Only in INKAR:", len(only_inkar))

print("\nGISCO only:")
print(only_gisco)

print("\nINKAR only:")
print(only_inkar)

Only in GISCO: 18
Only in INKAR: 19

GISCO only:
['darmstadt', 'dillingen a.d. donau', 'eisenach', 'friesland (de)', 'hagen', 'hamburg', 'kiel', 'lübeck', 'magdeburg', 'mühldorf a. inn', 'neumarkt i. d. opf.', 'neustadt a. d. aisch-bad windsheim', 'neustadt a. d. waldnaab', 'pfaffenhofen a. d. ilm', 'solingen', 'weiden i. d. opf', 'wiesbaden', 'wunsiedel i. fichtelgebirge']

INKAR only:
['darmstadt wissenschaftsstadt', 'dillingen a.d.donau', 'friesland', 'hagen der fernuniversität', 'hamburg freie und hansestadt', 'kassel documenta-', 'kiel landeshauptstadt', 'lübeck hansestadt', 'magdeburg landeshauptstadt', 'mühldorf a.inn', 'münchen landeshauptstadt', 'neumarkt i.d.opf.', 'neustadt a.d.aisch-bad windsheim', 'neustadt a.d.waldnaab', 'pfaffenhofen a.d.ilm', 'solingen klingenstadt', 'weiden i.d.opf.', 'wiesbaden landeshauptstadt', 'wunsiedel i.fichtelgebirge']


#### Inspect the remaining differences

In [81]:
# ==========================================================
# Remaining unmatched districts
# ==========================================================

display(
    gisco_clean[gisco_clean["clean_name"].isin(only_gisco)]
    .sort_values("clean_name")
)

display(
    inkar_clean[inkar_clean["clean_name"].isin(only_inkar)]
    .sort_values("clean_name")
)

,nuts3_id,nuts3_name,clean_name
162,DE711,"Darmstadt, Kreisfreie Stadt",darmstadt
132,DE277,Dillingen a.d. Donau,dillingen a.d. donau
399,DEG0N,"Eisenach, Kreisfreie Stadt",eisenach
233,DE94A,Friesland (DE),friesland (de)
284,DEA53,"Hagen, Kreisfreie Stadt",hagen
161,DE600,Hamburg,hamburg
364,DEF02,"Kiel, Kreisfreie Stadt",kiel
365,DEF03,"Lübeck, Kreisfreie Stadt",lübeck
351,DEE03,"Magdeburg, Kreisfreie Stadt",magdeburg
59,DE21G,Mühldorf a. Inn,mühldorf a. inn


,district_code,district_name,clean_name
116,6411,"Darmstadt, Wissenschaftsstadt",darmstadt wissenschaftsstadt
310,9773,Dillingen a.d.Donau,dillingen a.d.donau
53,3455,Friesland,friesland
106,5914,"Hagen, Stadt der FernUniversität",hagen der fernuniversität
15,2000,"Hamburg, Freie und Hansestadt",hamburg freie und hansestadt
135,6611,"Kassel, documenta-Stadt",kassel documenta-
1,1002,"Kiel, Landeshauptstadt",kiel landeshauptstadt
2,1003,"Lübeck, Hansestadt",lübeck hansestadt
366,15003,"Magdeburg, Landeshauptstadt",magdeburg landeshauptstadt
237,9183,Mühldorf a.Inn,mühldorf a.inn


#### Inspect remaining unmatched names exactly as stored

In [82]:
# ==========================================================
# Inspect remaining unmatched names exactly as stored
# ==========================================================

for name in sorted(only_gisco):
    print(repr(name))

print("\n" + "="*60 + "\n")

for name in sorted(only_inkar):
    print(repr(name))

'darmstadt'
'dillingen a.d. donau'
'eisenach'
'friesland (de)'
'hagen'
'hamburg'
'kiel'
'lübeck'
'magdeburg'
'mühldorf a. inn'
'neumarkt i. d. opf.'
'neustadt a. d. aisch-bad windsheim'
'neustadt a. d. waldnaab'
'pfaffenhofen a. d. ilm'
'solingen'
'weiden i. d. opf'
'wiesbaden'
'wunsiedel i. fichtelgebirge'


'darmstadt wissenschaftsstadt'
'dillingen a.d.donau'
'friesland'
'hagen der fernuniversität'
'hamburg freie und hansestadt'
'kassel documenta-'
'kiel landeshauptstadt'
'lübeck hansestadt'
'magdeburg landeshauptstadt'
'mühldorf a.inn'
'münchen landeshauptstadt'
'neumarkt i.d.opf.'
'neustadt a.d.aisch-bad windsheim'
'neustadt a.d.waldnaab'
'pfaffenhofen a.d.ilm'
'solingen klingenstadt'
'weiden i.d.opf.'
'wiesbaden landeshauptstadt'
'wunsiedel i.fichtelgebirge'


### Resolution of the 401 vs. 400 District Discrepancy

The compatibility assessment identified the source of the one-district difference between the GISCO NUTS-3 spatial dataset (401 districts) and the INKAR socio-economic datasets (400 districts).


| Dataset                 | District                   | Identifier                                                 |
| ----------------------- | -------------------------- | ---------------------------------------------------------- |
| **GISCO NUTS-3 (2021)** | Eisenach, Kreisfreie Stadt | **DEG0N**                                                  |
| **INKAR**               | Not listed separately      | Included within **Wartburgkreis (AGS 16063 / NUTS DEG0P)** |


#### Explanation

This discrepancy results from a genuine administrative boundary reform in the German state of Thüringen.

Until 30 June 2021, Eisenach existed as an independent Kreisfreie Stadt (urban district). On 1 July 2021, it was officially incorporated into the surrounding Wartburgkreis as part of the Thüringen district reorganization. Consequently, Eisenach ceased to exist as an independent district in the current German administrative structure.

The discrepancy arises because the two datasets reflect different administrative reference dates following the 2021 Thüringen district reform.

- GISCO NUTS 2021 still contains Eisenach (DEG0N) as a separate NUTS-3 region because the NUTS 2021 territorial classification was established before the July 2021 merger took effect. The corresponding Wartburgkreis is listed separately as DEG0P.
- INKAR, in contrast, reflects the post-merger German administrative structure and therefore reports 400 districts, with Eisenach incorporated into Wartburgkreis (AGS 16063 / NUTS DEG0P).


### Next Step

The spatial compatibility assessment confirms that the validated NUTS-2 → NUTS-3 Spatial Crosswalk provides a reliable basis for harmonizing the climate datasets.

The next phase of the project will temporarily shift to the socio-economic datasets, where the missing years will be imputed and the datasets finalized. Afterward, an Administrative Crosswalk (AGS ↔ NUTS-3) will be constructed to reconcile the documented Eisenach/Wartburgkreis administrative change and establish a consistent 400-district framework.

Once both the socio-economic datasets and the Administrative Crosswalk have been completed, the validated Spatial Crosswalk will be used to harmonize the heatwave, flood (Rx5day), and drought datasets from the NUTS-2 regional level to Germany's NUTS-3 district level using area-weighted interpolation. The harmonized climate datasets will then be integrated with the socio-economic indicators to construct the unified Climate–ESG master dataset.

# Part II — Harmonize Climate Datasets

#### 2.1 Harmonize Heatwave Dataset

(code...)

#### 2.2 Harmonize Flood (Rx5day) Dataset

(code...)

#### 2.3 Harmonize Drought Dataset

(code...)

# Part III — Export Harmonized Climate Datasets


#### 3.1 Export Harmonized Heatwave Dataset

#### 3.2 Export Harmonized Flood (Rx5day) Dataset

#### 3.3 Export Harmonized Drought Dataset

#### 3.4 Export Combined Climate Dataset (Optional)